# 第4部 マイコンに載せる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asg1223/rcj-soccer-ml/blob/main/notebooks/04_on_device.ipynb)

このノートブックは、教材のページのコードを順番に並べたものです（`tools/build_notebooks.py` で自動生成）。
上のセルから順に `Shift + Enter` で実行してください。

In [ ]:
# Colab のときだけ、教材リポジトリを取ってきて読み込めるようにする
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone -q --depth 1 https://github.com/Asg1223/rcj-soccer-ml.git /content/rcj-soccer-ml
    sys.path.insert(0, "/content/rcj-soccer-ml")
print("Colab で実行中" if IN_COLAB else "ローカルで実行中")

## 4-1 推論の中身は掛け算と足し算：C で動かす

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part4-on-device/export-c/

### 手を動かす 1：学習して重みを取り出す

In [ ]:
import numpy as np
import keras
from rcjml import load_config
from rcjml.data import load_synthetic, split_by_session
from rcjml.nn import features, targets, evaluate, keras_loss, build_ballnet

cfg = load_config()
df = load_synthetic("ring12_large_normal")
train_all, test = split_by_session(df)
last = train_all.session.max()
train, val = train_all[train_all.session < last], train_all[train_all.session == last]

keras.utils.set_random_seed(0)
model = build_ballnet()                       # 12 → 64 → 64 → 4（3-2 と同じ）
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=keras_loss())
model.fit(features(train, cfg), targets(train),
          validation_data=(features(val, cfg), targets(val)),
          epochs=400, batch_size=64, verbose=0,
          callbacks=[keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)])

X_test = features(test, cfg)
P_keras = model.predict(X_test, verbose=0)
print(evaluate(P_keras, test))

In [ ]:
from rcjml.export import layers_from_keras, forward

layers = layers_from_keras(model)
for k, L in enumerate(layers):
    print(f"層 {k}: 重み {L.W.shape}, バイアス {L.b.shape}, 活性化 {L.activation}")

# NumPy で同じ計算をして、Keras と同じ答えになるか確かめる
P_numpy = forward(layers, X_test)
print("Keras との差（いちばん大きいもの）:", np.abs(P_keras - P_numpy).max())

### 手を動かす 2：C のヘッダに書き出す

In [ ]:
from rcjml.config import PACKAGE_ROOT
from rcjml.export import export_to_sketch

sketch = PACKAGE_ROOT / "firmware" / "sensor-arduino" / "ballnet_infer"
files = export_to_sketch(model, sketch, allow_public=True)
for f in files:
    print(f, f.stat().st_size, "bytes")

### 手を動かす 4：PC で C の答え合わせ

In [ ]:
import shutil, subprocess

if shutil.which("gcc") is None:
    print("gcc がないので、この確かめはとばします（Windows なら Colab で試せます）")
else:
    np.savetxt(sketch / "x_test.txt", X_test[:200])
    (sketch / "check.c").write_text(r'''
#include <stdio.h>
#include "ballnet_model.h"
int main(void) {
  FILE* f = fopen("x_test.txt", "r"); float x[BALLNET_N_IN], y[BALLNET_N_OUT];
  while (1) {
    for (int i = 0; i < BALLNET_N_IN; i++) if (fscanf(f, "%f", &x[i]) != 1) return 0;
    ballnet_forward(x, y);
    printf("%.7e %.7e %.7e %.7e\n", y[0], y[1], y[2], y[3]);
  }
}''')
    subprocess.run(["gcc", "-O2", "-o", "check", "check.c", "-lm"], cwd=sketch, check=True)
    out = subprocess.run(["./check"], cwd=sketch, capture_output=True, text=True).stdout
    P_c = np.array([[float(v) for v in line.split()] for line in out.splitlines()])
    print("C と Keras の差（いちばん大きいもの）:", np.abs(P_c - P_keras[:200]).max())
    for name in ["x_test.txt", "check.c", "check"]:
        (sketch / name).unlink()

## 4-2 メモリと速さを測る

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part4-on-device/measure/

### 手を動かす 1：PC で見積もる

In [ ]:
from rcjml.export import layers_from_keras

layers = layers_from_keras(model)
n_params = sum(L.W.size + L.b.size for L in layers)
n_mac = sum(L.W.size for L in layers)          # かけて足す回数
width = max(L.W.shape[1] for L in layers)
print(f"重みとバイアス : {n_params} 個 = {n_params * 4 / 1024:.1f} KB（float）")
print(f"かけて足す回数 : {n_mac} 回 / 1 推論")
print(f"作業用の RAM   : {width * 2 * 4} バイト")

In [ ]:
import pandas as pd
rows = []
for hidden in [(16,), (32,), (32, 32), (64, 64), (128, 128)]:
    sizes = [12, *hidden, 4]
    params = sum(a * b + b for a, b in zip(sizes[:-1], sizes[1:]))
    macs = sum(a * b for a, b in zip(sizes[:-1], sizes[1:]))
    rows.append({"hidden": hidden, "params": params, "flash_KB(float)": round(params * 4 / 1024, 1),
                 "flash_KB(int8)": round(params / 1024, 1), "MAC": macs})
pd.DataFrame(rows)

## BallNet bench

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part4-on-device/measure/

## 4-3 量子化：重みを int8 にして小さくする

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part4-on-device/quantization/

### 手を動かす 1：自分で量子化してみる

In [ ]:
from rcjml.export import quantize, dequantize

W = layers[0].W                       # 1 層目の重み（12 × 64）
q, scale = quantize(W)
print("scale:", scale)
print("元の重み  :", W[0, :5].round(4))
print("int8      :", q[0, :5])
print("戻した重み:", dequantize(q, scale)[0, :5].round(4))
print("いちばん大きいずれ:", np.abs(W - dequantize(q, scale)).max())

### 手を動かす 2：成績がどう変わるか

In [ ]:
P_float = forward(layers, X_test)
P_int8 = forward(layers, X_test, quantized=True)

for name, P in [("float", P_float), ("int8 weights", P_int8)]:
    r = evaluate(P, test)
    print(f"{name:13s} 角度 {r['angle_MAE']:.2f}°  距離 {r['dist_rel_median']:.3f}  保持 {r['hold_acc']:.4f}")

n_w = sum(L.W.size for L in layers)
n_b = sum(L.b.size for L in layers)
print(f"重みの大きさ: float {(n_w + n_b) * 4} バイト → int8 {n_w + n_b * 4} バイト")

### 手を動かす 3：int8 のヘッダを書き出す

In [ ]:
from rcjml.config import PACKAGE_ROOT
from rcjml.export import export_to_sketch
export_to_sketch(model, PACKAGE_ROOT / "firmware" / "sensor-arduino" / "ballnet_infer",
                 quantized=True, allow_public=True)

## 4-4 TensorFlow Lite Micro で動かす

説明は教材のページを見てください：https://asg1223.github.io/rcj-soccer-ml/part4-on-device/tflite-micro/

### 手を動かす 1：.tflite に変換する

In [ ]:
import tensorflow as tf

# そのまま（float）
float_tflite = tf.lite.TFLiteConverter.from_keras_model(model).convert()

# 全部 int8。途中の値の範囲を測るために、学習用データの一部（代表データ）を渡す
def representative_data():
    X = features(train, cfg)
    for i in range(0, len(X), 20):
        yield [X[i:i + 1]]

conv = tf.lite.TFLiteConverter.from_keras_model(model)
conv.optimizations = [tf.lite.Optimize.DEFAULT]
conv.representative_dataset = representative_data
conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
int8_tflite = conv.convert()

print("float:", len(float_tflite), "バイト")
print("int8 :", len(int8_tflite), "バイト")

### 手を動かす 2：PC で成績を確かめる

In [ ]:
def run_tflite(model_bytes, X):
    it = tf.lite.Interpreter(model_content=model_bytes)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    P = []
    for x in X:
        it.set_tensor(inp["index"], x[None, :].astype(np.float32))
        it.invoke()
        P.append(it.get_tensor(out["index"])[0])
    return np.array(P)

for name, b in [("float", float_tflite), ("int8", int8_tflite)]:
    r = evaluate(run_tflite(b, X_test), test)
    print(f"{name:6s} 角度 {r['angle_MAE']:.2f}°  距離 {r['dist_rel_median']:.3f}  保持 {r['hold_acc']:.4f}")

### 手を動かす 3：C の配列にする

In [ ]:
from rcjml.config import PACKAGE_ROOT
from rcjml.export import tflite_to_c

path = tflite_to_c(int8_tflite, PACKAGE_ROOT / "firmware" / "cores3" / "ballnet_tflm" / "ballnet_tflite.h")
print(path, path.stat().st_size, "バイト（文字にしたので元より大きい。書き込まれるのは元の大きさ）")